## Pre Processing

**Import Necessary Libraries**

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')
pd.pandas.set_option("display.max_columns", None)

# Create Dataframe
df = pd.read_csv(r"EasyVisa.csv")
# Print shape of dataset
print(df.shape)

**Data Cleaning**

In [ ]:
#check null values
def checkmissingvalues(df):
    missingcounts = df.isnull().sum()
    missingcounts_columns = missingcounts[missingcounts > 0].index
    count = 0
    
    if len(missingcounts_columns) > 0 :
        print("Missing Columns are :\n")
        for i in missingcounts_columns:
            count += 1
            print(f"{count}-{i}")
            print(i,np.round(df[i].isnull().mean()*100,5), '% missing values')
    else:
        print("There is no missing value in Dataframe")

checkmissingvalues(df)

**Check Duplicated Values**

In [ ]:
df.duplicated().sum()

**Remove case_id from the dataset as it cannot used in Model Training**

In [ ]:
df.drop(columns=['case_id'],inplace=True)

In [ ]:
df.info()

## Feature Engineering

**Feature Extraction**

In [ ]:
df.head()

In [ ]:
from datetime import date
todays_date = date.today()
current_year = todays_date.year


In [ ]:
df['company_age'] = current_year- df['yr_of_estab'] 
df[['company_age' , 'yr_of_estab']]

In [ ]:
df.drop(columns=['yr_of_estab'],inplace=True)

In [ ]:
df.info()

**Types of Features**

In [ ]:
#numeric features
numerical_features = [i for i in df.columns if df[i].dtype != "O"]
print(f"No. Of {len(numerical_features)} numerical features are : \n{numerical_features}")

In [ ]:
#categorical features
categorical_features = [i for i in df.columns if df[i].dtype == "O"]
print(f"No. Of {len(categorical_features)} categorical features are : \n{categorical_features}")

In [ ]:
#discrete numerical_features

discrete_features = [i for i in numerical_features if len(df[i].unique()) <=25]
print(f"No. Of {len(discrete_features)} discrete numerical features are : \n{discrete_features}")

In [ ]:
#continuous numerical_features

continuous_features = [i for i in numerical_features if len(df[i].unique()) >= 25]
print(f"No. Of {len(continuous_features)} continuous numerical features are : \n{continuous_features}")

## Splitting the data

In [ ]:
X = df.drop(['case_status'],axis=1)
y = df['case_status']

In [ ]:
X

In [ ]:
y.head()

**Manually Encoding Target Column**

In [ ]:
y = np.where(y=='Denied',0,1)

In [ ]:
y

### Feature Transformation

In [ ]:
plt.figure(figsize=(12,6))

for i,col in enumerate(['no_of_employees', 'prevailing_wage', 'company_age']):
    plt.subplot(2,2,i+1)
    sns.histplot(x=X[col],color='indianred')
    plt.xlabel(col)
    plt.tight_layout()


* No of employees and Company age column is skewed
* Apply a power transform featurewise to make data more Gaussian-like.
#### Power transforms are a family of parametric, monotonic transformations that are applied to make data more Gaussian-like. This is useful for modeling issues related to heteroscedasticity (non-constant variance), or other situations where normality is desired.

* Currently, PowerTransformer supports the Box-Cox transform and the Yeo-Johnson transform.

**Checking Skewness**

__What is Skewness ?__

* Skewness refers to a distortion or asymmetry that deviates from the symmetrical bell curve, or normal distribution, in a set of data. If the curve is shifted to the left or to the right, it is said to be skewed. Skewness can be quantified as a representation of the extent to which a given distribution varies from a normal distribution. A normal distribution has a skew of zero

In [ ]:
X[continuous_features].skew(axis=0, skipna=True)

* Positiviely Skewed : company_age, no_of_employees.
* We can handle outliers and then check the skewness.

### Apply Power Transformer to Check if it can reduces the outliers

In [ ]:
from sklearn.preprocessing import PowerTransformer
pt = PowerTransformer(method='yeo-johnson')
transform_features = ['company_age', 'no_of_employees']
X_copy = pt.fit_transform(X[transform_features])

In [ ]:
X_copy = pd.DataFrame(X_copy, columns=transform_features)

In [ ]:
plt.figure(figsize=(12, 5))
for i, col in enumerate(transform_features):
    plt.subplot(1, 2, i+1)
    sns.histplot(x=X_copy[col], color='indianred')
    plt.xlabel(col)
    plt.tight_layout()

**Checking Skewness**

In [ ]:
X_copy[transform_features].skew(axis=0,skipna=True)

* Here Yeo-Johnson is used and it supports both positive or negative data for transformation.
* So Power Transformer with yeo-johnson can be used.

In [ ]:
for i in categorical_features:
    print(i,':',df[i].nunique())

### Feature Encoding and Scaling
* One Hot Encoding for Columns which had lesser unique values and not ordinal

* One hot encoding is a process by which categorical variables are converted into a form that could be provided to ML algorithms to do a better job in prediction.

**Ordinal Encoding for Columns which has many unique categories**

* Ordinal encoding is used here as label encoder is supported for column transformer.
* Ordinal encoding is used for Ordinal Variable. Variable comprises a finite set of discrete values with a ranked ordering between values.

### Standard Scaler

* Standardize features by removing the mean and scaling to unit variance.

### Power Transformer

Power transforms are a technique for transforming numerical input or output variables to have a Gaussian or more-Gaussian-like probability distribution.

In [ ]:
num_features = list(X.select_dtypes(exclude="object").columns)
num_features

## Pre-Processing Pipeline

In [ ]:
ordinal_col = ['has_job_experience','requires_job_training','full_time_position','education_of_employee']
ohe_col = ['continent','unit_of_wage','region_of_employment']
transform_col = ['no_of_employees', 'company_age']


from sklearn.preprocessing import StandardScaler,OneHotEncoder,OrdinalEncoder,PowerTransformer
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

'''Initialize Object'''
numeric_transformer = StandardScaler()
oh_transformer = OneHotEncoder()
ordinal_encoder = OrdinalEncoder()

transform_pipe = Pipeline(steps=[
    ('transformer', PowerTransformer(method='yeo-johnson'))
])

preprocessor = ColumnTransformer(
    [
        ("OneHotEncoder", oh_transformer, ohe_col),
        ("Ordinal_Encoder", ordinal_encoder, ordinal_col),
        ("Transformer", transform_pipe, transform_col),
        ("StandardScaler", numeric_transformer, num_features)
    ]
)

preprocessor


In [ ]:
X=preprocessor.fit_transform(X)

**Imbalance Data Issue**

In [ ]:
from imblearn.combine import SMOTETomek, SMOTEENN

# Resampling the minority class. The strategy can be changed as required.
smt = SMOTEENN(random_state=42,sampling_strategy='minority' )
# Fit the model to generate the data.
X_res, y_res = smt.fit_resample(X, y)

In [ ]:
from sklearn.model_selection import  train_test_split
# separate dataset into train and test
X_train, X_test, y_train, y_test = train_test_split(X_res,y_res,test_size=0.2,random_state=42)
X_train.shape, X_test.shape

In [ ]:
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report,ConfusionMatrixDisplay, \
                            precision_score, recall_score, f1_score, roc_auc_score,roc_curve 
from xgboost import XGBClassifier
from catboost import CatBoostClassifier

In [ ]:
def evaluate_clf(true, predicted):
    acc = accuracy_score(true, predicted) # Calculate Accuracy
    f1 = f1_score(true, predicted) # Calculate F1-score
    precision = precision_score(true, predicted) # Calculate Precision
    recall = recall_score(true, predicted)  # Calculate Recall
    roc_auc = roc_auc_score(true, predicted) #Calculate Roc
    return acc, f1 , precision, recall, roc_auc

In [ ]:
models = {
    "Random Forest": RandomForestClassifier(),
    "Decision Tree": DecisionTreeClassifier(),
    "Gradient Boosting": GradientBoostingClassifier(),
    "Logistic Regression": LogisticRegression(),
     "K-Neighbors Classifier": KNeighborsClassifier(),
    "XGBClassifier": XGBClassifier(), 
     "CatBoosting Classifier": CatBoostClassifier(verbose=False),
     "Support Vector Classifier": SVC(),
    "AdaBoost Classifier": AdaBoostClassifier()

}
models

In [ ]:
# Create a function which can evaluate models and return a report 
def evaluate_models(X, y, models):
    '''
    This function takes in X and y and models dictionary as input
    It splits the data into Train Test split
    Iterates through the given model dictionary and evaluates the metrics
    Returns: Dataframe which contains report of all models metrics with cost
    '''
    # separate dataset into train and test
    X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.2,random_state=42)
    
    models_list = []
    accuracy_list = []
    auc= []
    
    for i in range(len(list(models))):
        model = list(models.values())[i]
        model.fit(X_train, y_train) # Train model

        # Make predictions
        y_train_pred = model.predict(X_train)
        y_test_pred = model.predict(X_test)

        # Training set performance
        model_train_accuracy, model_train_f1,model_train_precision,\
        model_train_recall,model_train_rocauc_score=evaluate_clf(y_train ,y_train_pred)


        # Test set performance
        model_test_accuracy,model_test_f1,model_test_precision,\
        model_test_recall,model_test_rocauc_score=evaluate_clf(y_test, y_test_pred)

        print(list(models.keys())[i])
        models_list.append(list(models.keys())[i])

        print('Model performance for Training set')
        print("- Accuracy: {:.4f}".format(model_train_accuracy))
        print('- F1 score: {:.4f}'.format(model_train_f1)) 
        print('- Precision: {:.4f}'.format(model_train_precision))
        print('- Recall: {:.4f}'.format(model_train_recall))
        print('- Roc Auc Score: {:.4f}'.format(model_train_rocauc_score))

        print('----------------------------------')

        print('Model performance for Test set')
        print('- Accuracy: {:.4f}'.format(model_test_accuracy))
        accuracy_list.append(model_test_accuracy)
        print('- F1 score: {:.4f}'.format(model_test_f1))
        print('- Precision: {:.4f}'.format(model_test_precision))
        print('- Recall: {:.4f}'.format(model_test_recall))
        print('- Roc Auc Score: {:.4f}'.format(model_test_rocauc_score))
        auc.append(model_test_rocauc_score)
        print('='*35)
        print('\n')
        
    report=pd.DataFrame(list(zip(models_list, accuracy_list)), columns=['Model Name', 'Accuracy']).sort_values(by=['Accuracy'], ascending=False)
        
    return report

### Model Training

In [ ]:
base_model_report =evaluate_models(X=X_res, y=y_res, models=models)

### Model Reports

In [ ]:
base_model_report

## HyperParameter tuning

In [ ]:
#Initialize few parameter for Hyperparamter tuning
xgboost_params = {
    'max_depth':range(3,10,2),
    'min_child_weight':range(1,6,2)
}

rf_params = {
    "max_depth": [10, 12, None, 15, 20],
    "max_features": ['sqrt', 'log2', None],
    "n_estimators": [10, 50, 100, 200]
}

knn_params = {
    "algorithm": ['auto', 'ball_tree', 'kd_tree','brute'],
    "weights": ['uniform', 'distance'],
    "n_neighbors": [3, 4, 5, 7, 9],
}

In [ ]:
# Models list for Hyperparameter tuning
randomcv_models = [
    ('XGBoost', XGBClassifier(), xgboost_params),
    ("RF", RandomForestClassifier(), rf_params),
    ("KNN", KNeighborsClassifier(), knn_params)
]

In [ ]:
from sklearn.model_selection import RandomizedSearchCV

model_param = {}
for name, model, params in randomcv_models:
    random = RandomizedSearchCV(estimator=model,
                                   param_distributions=params,
                                   n_iter=100,
                                   cv=3,
                                   verbose=2, 
                                   n_jobs=-1)
    random.fit(X_res, y_res)
    model_param[name] = random.best_params_

for model_name in model_param:
    print(f"---------------- Best Params for {model_name} -------------------")
    print(model_param[model_name])

In [ ]:
model_param

### Retraining Model with Hyper Parameters

In [ ]:
from sklearn.metrics import roc_auc_score,roc_curve
best_models = {
    "Random Forest Classifier": RandomForestClassifier(**model_param['RF']),
    "KNeighborsClassifier": KNeighborsClassifier(**model_param['KNN']),
    "XGBClassifier": XGBClassifier(**model_param['XGBoost'],n_jobs=-1),
}
tuned_report =evaluate_models(X=X_res, y=y_res, models=best_models)

In [ ]:
tuned_report

In [ ]:
best_model = KNeighborsClassifier(**model_param['KNN'])
best_model = best_model.fit(X_train,y_train)
y_pred = best_model.predict(X_test)
score = accuracy_score(y_test,y_pred)
cr = classification_report(y_test,y_pred)

print("FINAL MODEL 'KNN'")
print ("Accuracy Score value: {:.4f}".format(score))
print (cr)

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay
ConfusionMatrixDisplay.from_estimator(best_model, X_test, y_test)

## Best Model is K-Nearest Neighbor(KNN) with Accuracy 96.19%